In [ ]:

from pymongo import MongoClient

client = MongoClient(MONGO_URL)
db = client["your_db"]
col = db["your_collection"]

### Preparation

In [ ]:
import pandas as pd 
import numpy as np
import os
from pathlib import Path
from datetime import datetime

save_path = DATA / "???"

now = datetime.now().strftime("%Y%m%d_%H%M%S")
# loading data from MongoDB
cursor = col.find(
    {},
    {"_id": 0, 
     "productid": 1, 
     "text_vector": 1,
     "image_vector": 1}
    )

docs = list(cursor)
docs = sorted(docs, key=lambda x: x["productid"])

product_ids = np.array([d["productid"] for d in docs], dtype=np.int32)
text_emb = np.array([d["text_vector"] for d in docs], dtype=np.float32)
# image_emb= np.array([d["iamge_vector"] for d in docs], dtype=np.float32)

id_to_index = {pid: i for i, pid in enumerate(product_ids)}
# desig_to_pid nötig ???

# # Version B
# text_emb = np.array([doc["text_vector"] for doc in col.find({}, 
#             {"_id":0, "text_vector":1})], dtype=np.float32)

# img_emb = np.array([doc["image_vector"] for doc in col.find({}, 
#             {"_id":0, "image_vector":1})], dtype=np.float32)

now = datetime.now().strftime("%Y%m%d_%H%M%S")

# array check 
for name, ar in zip(["product_ids",
            "text_emb", 
            # "img_emb"
            ],
            [product_ids,
            text_emb, 
            # img_emb
            ]):
    
    print(f"{'='*30}\n--- CHECKING ARRAY'{name}' ---\n{'='*30}")
    print(f"Type:\t-->{type(ar)}\nShape:\t-->{ar.shape}\nDtype:\t-->{ar.dtype}\n")
    print(f"First 3 rows of array '{name}':\n{ar[:3]}\n")

    try:
        np.save(os.path.join(save_path, f"{ar}_{now}.npy"), ar)
        print(f"Array '{name}' saved successfully at {save_path}/{ar}.npy\n")
    except Exception as e:
        print(f"Error saving array '{name}': {e}\n")    

    # if not isinstance(ar, np.ndarray):
    #     raise ValueError("Input is not a numpy array")
    # if ar.ndim != 2:
    #     raise ValueError("Input array is not 2-dimensional")


In [ ]:
# if necessary, normalize vectors
def l2_norm(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

image_emb = l2_norm(img_emb)
# txt_emb = l2_norm(text_emb) 

# combining vectors to recommender system matrix
alpha = 0.3          # weight for image vector
beta = 1- alpha      # weight for text vector

combined = np.hstack([alpha * image_emb, beta * txt_emb])
combined_norm = l2_norm(.astype("float32"))
now = datetime.now().strftime("%Y%m%d_%H%M%S")

# array check 
for name, ar in zip(["combined"], [combined]):
    
    print(f"{'='*30}\n--- CHECKING ARRAY'{name}' ---\n{'='*30}")
    print(f"Type:\t-->{type(ar)}\nShape:\t-->{ar.shape}\nDtype:\t-->{ar.dtype}\n")
    print(f"First 3 rows of array '{name}':\n{ar[:3]}\n")

    try:
        np.save(os.path.join(save_path, f"{ar}_{now}.npy"), ar)
        print(f"Array '{name}' saved successfully at {save_path}/{ar}.npy\n")
    except Exception as e:
        print(f"Error saving array '{name}': {e}\n")  


In [ ]:
# "TRAINING": calculating similarity matrices
# (A) KNN with cosine similarity
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(metric="cosine", 
                       n_neighbors=5)
sm_knn = knn.fit(combined)

# # (C)Cosine similarity (from sklearn)
# from sklearn.metrics.pairwise import cosine_similarity

# sm_cosine = cosine_similarity(combined)

# (B) FAISS library
import faiss

d = combined_norm.shape[1]
index = faiss.IndexFlatIP(d)  # Using Inner Product for Cosine Similarity
index.add(combined_norm)       # Adding normalized vectors to the index



In [ ]:
# "PREDICTION": querying similarity matrices
# Example: querying for a specific product ID (pid) and number of neighbors (n_neighbors)
query_pid = 123456  # Example product ID
query_index = id_to_index[query_pid]

n_neighbors = 5

# (A) KNN with cosine similarity
distances, indices = sm_knn.kneighbors(combined[0:1], n_neighbors=n_neighbors)
recomm_ids_knn = product_ids[indices[:n_neighbors]]

print("KNN Indices:", indices)
print("KNN Distances:", distances)
print("Recommended Product_ids:\n", recomm_ids_knn)  ## anstelledessen Ausgabe von 'designation' oder Aufruf Image!?

# (B) FAISS
D, I = index.search(combined_norm[0:1], n_neighbors)  # Searching
recomm_ids_faiss = product_ids[I[:n_neighbors]]

print("FAISS Indices:", I)
print("FAISS Distances:", D)    
print("Recommended Product_ids:\n", recomm_ids_faiss)   ## anstelledessen Ausgabe von 'designation' oder Aufruf Image!?                   